# 08 · Gürültü Giderme: Metni En Az Bozan Yöntem Hangisi?

**15 Ekim danışman görüşmesi (istek 6):** Median filtre gürültüyle birlikte yazıyı da siliyor. Asistan **Fourier dönüşümü + eşik değeri** önerdi.
Danışman: *"Hangisini kullanırsan kullan biraz yiyecek; minimum yemek hangisinde olacak?"*

Bu notebook aynı belgelere farklı gürültü giderme yöntemleri uygular, her birinde OCR yapar ve **ne kadar metin kaybettiğini / kazandığını** ölçer.

| Yöntem | Fikir |
|---|---|
| A · Ham görüntü | Karşılaştırma noktası (şu an kullanılan) |
| B · Median 3×3 | Her pikseli komşularının ortancasıyla değiştirir; tuz-biber gürültüsünü siler, ince çizgileri de inceltir |
| C · Gauss 3×3 | Hafif bulanıklaştırma |
| D · Non-local means | Benzer dokulu bölgelerin ortalaması; kenarları korumaya çalışır |
| E · Fourier alçak geçiren | Yüksek frekansları (ince ayrıntı, nokta gürültüsü) keser |
| F · Fourier eşik (%50) | Genliği küçük olan frekans katsayılarının yarısı silinir ("küçük sinyalleri kırma") |
| G · Fourier eşik (%90) | Katsayıların %90'ı silinir, sadece en güçlü %10 kalır |
| H · Küçük leke silme | 6 pikselden küçük siyah noktalar (harf olamayacak kadar küçük) silinir, geri kalan görüntüye dokunulmaz |

**Ölçütler** (RVL-CDIP'te belgelerin doğru metni olmadığı için dolaylı):

| Ölçüt | Anlamı |
|---|---|
| OCR güveni | Tesseract'ın kelimelerden ortalama ne kadar emin olduğu |
| Sözlük oranı | Okunan kelimelerin yüzde kaçı gerçek İngilizce kelime |
| Korunan kelime (%) | Ham görüntüde okunan **doğru** kelimelerin yüzde kaçı bu yöntemden sonra da okunuyor → **ne kadar yediği** |
| Kazanılan kelime | Ham görüntüde okunamayıp bu yöntemle okunan doğru kelime sayısı |
| Koyu piksel kaybı (%) | Görüntüdeki yazı (koyu) piksellerinin yüzde kaçı silindi |

Örneklem: 03'teki pilot deneyle aynı → her sınıftan 10 **validation** belgesi. Süre ~10–15 dk.

## 1. Kurulum

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract opencv-python-headless nltk symspellpy

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys, json, importlib, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
CFG_FILE = ROOT / "siniflar.json"
CLASSES = json.loads(CFG_FILE.read_text())["CLASSES"] if CFG_FILE.exists() else ["email", "form", "invoice", "resume"]
print("Sınıflar:", CLASSES)

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_core.py
import cv2
import numpy as np
import pytesseract


def estimate_skew(gray, max_angle=5.0, step=0.25):
    """Projection-profile skew estimate: the angle whose row sums are sharpest."""
    small = cv2.resize(gray, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    th = cv2.medianBlur(th, 3)  # drop isolated noise pixels
    h, w = th.shape
    best_angle, best_score = 0.0, -1.0
    for a in np.arange(-max_angle, max_angle + 1e-9, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        r = cv2.warpAffine(th, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
        score = float(np.var(r.sum(axis=1, dtype=np.float64)))
        if score > best_score:
            best_angle, best_score = float(a), score
    return best_angle


def deskew(gray):
    """Rotate the page so text lines are horizontal."""
    angle = estimate_skew(gray)
    if abs(angle) < 0.3:
        return gray, 0.0
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def preprocess(img_bgr_or_gray, upscale_to=2000, do_deskew=True, binarize=True):
    """Grayscale -> upscale (RVL-CDIP is max 1000px) -> denoise -> deskew -> Otsu."""
    img = img_bgr_or_gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()
    gray = cv2.medianBlur(gray, 3)  # salt-and-pepper scan noise, before upscaling
    h, w = gray.shape
    scale = upscale_to / max(h, w)
    if scale > 1.0:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    angle = 0.0
    if do_deskew:
        gray, angle = deskew(gray)
    if binarize:
        _, gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return gray, angle


def run_ocr(img, lang="eng", psm=3, min_conf=0):
    """Tesseract OCR -> words, normalized boxes (0-1000, LayoutLM format), line text."""
    h, w = img.shape[:2]
    d = pytesseract.image_to_data(img, lang=lang, config=f"--oem 1 --psm {psm}",
                                  output_type=pytesseract.Output.DICT)
    words, boxes, confs, lines = [], [], [], {}
    for i, txt in enumerate(d["text"]):
        txt = txt.strip()
        conf = float(d["conf"][i])
        if not txt or conf < min_conf:
            continue
        x, y, bw, bh = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
        words.append(txt)
        confs.append(conf)
        boxes.append([int(1000 * x / w), int(1000 * y / h),
                      int(1000 * (x + bw) / w), int(1000 * (y + bh) / h)])
        key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
        lines.setdefault(key, []).append(txt)
    text = "\n".join(" ".join(v) for _, v in sorted(lines.items()))
    return {
        "text": text,
        "words": words,
        "boxes": boxes,
        "confs": confs,
        "n_words": len(words),
        "mean_conf": float(np.mean(confs)) if confs else 0.0,
    }


def detect_table(gray):
    """Morfolojik işlemlerle yatay/dikey çizgileri bulup belgede tablo olup olmadığını tahmin eder.

    OCR tabloları bozarak okuduğu için (danışman/asistan görüşmesi) tablo bilgisi
    görüntüden ayrıca çıkarılır ve sınıflandırmada ek özellik olarak kullanılabilir.
    """
    img = gray if gray.ndim == 2 else cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    h, w = img.shape
    _, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(10, w // 25), 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(10, h // 25)))
    horiz = cv2.morphologyEx(th, cv2.MORPH_OPEN, hk)
    vert = cv2.morphologyEx(th, cv2.MORPH_OPEN, vk)
    n_h = len(cv2.findContours(horiz, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    n_v = len(cv2.findContours(vert, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    inter = cv2.bitwise_and(cv2.dilate(horiz, np.ones((3, 3), np.uint8)), cv2.dilate(vert, np.ones((3, 3), np.uint8)))
    n_x = len(cv2.findContours(inter, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    line_ratio = float((horiz > 0).mean() + (vert > 0).mean())
    return {
        "n_hlines": n_h,
        "n_vlines": n_v,
        "n_intersections": n_x,
        "line_ratio": round(line_ratio, 5),
        "has_table": bool(n_h >= 3 and n_v >= 2 and n_x >= 4),
    }


In [ ]:
sys.path.insert(0, str(ROOT))
import ocr_core; importlib.reload(ocr_core)
from ocr_core import run_ocr

import nltk
try:
    nltk.download("words", quiet=True)
    from nltk.corpus import words as nltk_words
    VOCAB = {w.lower() for w in nltk_words.words()}          # 03'teki pilot deneyle aynı sözlük
except Exception as e:
    print("nltk sözlüğü yüklenemedi:", e); VOCAB = set()
if not VOCAB:                                               # yedek: SymSpell'in İngilizce sıklık sözlüğü
    from importlib import resources
    VOCAB = {l.split()[0] for l in (resources.files("symspellpy") / "frequency_dictionary_en_82_765.txt").read_text().splitlines() if l.strip()}
print("Sözlük:", len(VOCAB), "kelime")

def dict_words(text):
    return [t.lower() for t in re.findall(r"[A-Za-z]+", text or "") if len(t) >= 3]

def dict_ratio(text):
    toks = dict_words(text)
    return sum(t in VOCAB for t in toks) / len(toks) if toks else 0.0

# Örneklem: 03'teki pilot deneyle aynı seçim (her sınıftan 10 validation belgesi, random_state=0)
parts = [pd.read_csv(DATA / f) for f in ("metadata.csv", "metadata_aday.csv") if (DATA / f).exists()]
meta = pd.concat(parts, ignore_index=True).drop_duplicates("path")
meta = meta[meta.label.isin(CLASSES)]
sample = pd.concat([g.sample(min(10, len(g)), random_state=0) for _, g in meta[meta.split == "validation"].groupby("label")])
print(len(sample), "belge"); sample.label.value_counts()

## 2. Gürültü giderme yöntemleri

In [ ]:
def median3(im):
    return cv2.medianBlur(im, 3)

def gauss3(im):
    return cv2.GaussianBlur(im, (3, 3), 0)

def nlm(im):
    return cv2.fastNlMeansDenoising(im, None, h=10, templateWindowSize=7, searchWindowSize=21)

def fourier_lowpass(im, keep=0.35):
    """Merkezden `keep` oranındaki yarıçap içindeki (düşük) frekanslar kalır, yüksek frekanslar kesilir."""
    f = np.fft.fftshift(np.fft.fft2(im.astype(np.float32)))
    h, w = im.shape
    yy, xx = np.ogrid[:h, :w]
    r = keep * min(h, w) / 2
    f[(yy - h // 2) ** 2 + (xx - w // 2) ** 2 > r * r] = 0
    return np.clip(np.real(np.fft.ifft2(np.fft.ifftshift(f))), 0, 255).astype(np.uint8)

def fourier_threshold(im, q):
    """Genliği en küçük `q` oranındaki frekans katsayıları sıfırlanır (küçük sinyalleri kırma)."""
    f = np.fft.fft2(im.astype(np.float32))
    mag = np.abs(f)
    f[mag < np.quantile(mag, q)] = 0
    return np.clip(np.real(np.fft.ifft2(f)), 0, 255).astype(np.uint8)

def remove_specks(im, max_area=6):
    """Harf olamayacak kadar küçük (≤ max_area piksel) koyu noktalar beyaza çevrilir; diğer piksellere dokunulmaz."""
    _, dark = cv2.threshold(im, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(dark, connectivity=8)
    small = np.where(stats[:, cv2.CC_STAT_AREA] <= max_area)[0]
    small = small[small != 0]
    out = im.copy()
    out[np.isin(lab, small)] = 255
    return out

METHODS = {
    "A · ham görüntü": lambda im: im,
    "B · median 3×3": median3,
    "C · Gauss 3×3": gauss3,
    "D · non-local means": nlm,
    "E · Fourier alçak geçiren": fourier_lowpass,
    "F · Fourier eşik %50": lambda im: fourier_threshold(im, 0.5),
    "G · Fourier eşik %90": lambda im: fourier_threshold(im, 0.9),
    "H · küçük leke silme": remove_specks,
}

In [ ]:
# Örnek: en çok küçük leke içeren belge üzerinde yöntemlerin görünümü (yakınlaştırılmış bölge)
def n_specks(im):
    _, dark = cv2.threshold(im, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    _, _, st, _ = cv2.connectedComponentsWithStats(dark, connectivity=8)
    return int((st[1:, cv2.CC_STAT_AREA] <= 6).sum())

imgs = {r.path: cv2.imread(str(DATA / r.path), cv2.IMREAD_GRAYSCALE) for r in sample.itertuples()}
noisy_path = max(imgs, key=lambda p: n_specks(imgs[p]))
im0 = imgs[noisy_path]
h, w = im0.shape
y0, x0 = h // 4, w // 6
crop = (slice(y0, y0 + h // 4), slice(x0, x0 + w // 2))
fig, axes = plt.subplots(2, 4, figsize=(18, 7.5))
for ax, (name, fn) in zip(axes.ravel(), METHODS.items()):
    ax.imshow(fn(im0)[crop], cmap="gray", vmin=0, vmax=255); ax.set_title(name, fontsize=11); ax.axis("off")
plt.suptitle(f"Aynı bölge, farklı yöntemler · {Path(noisy_path).name} ({n_specks(im0)} küçük leke)", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "gurultu_ornek.png", dpi=110); plt.show()

## 3. Her yöntemle OCR

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def run_one(args):
    path, label, name = args
    im = imgs[path]
    out = METHODS[name](im)
    o = run_ocr(out)
    _, dark = cv2.threshold(im, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    t = cv2.threshold(im, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0]
    lost = float(((out > t) & (dark > 0)).sum() / max((dark > 0).sum(), 1))
    return {"belge": path, "sınıf": label, "yöntem": name, "güven": o["mean_conf"], "kelime": o["n_words"],
            "sözlük_oranı": dict_ratio(o["text"]), "_words": {x for x in dict_words(o["text"]) if x in VOCAB},
            "koyu_piksel_kaybı": 100 * lost}

jobs = [(r.path, r.label, m) for r in sample.itertuples() for m in METHODS]
t0 = time.time()
with ThreadPoolExecutor(max_workers=2) as pool:
    res = list(tqdm(pool.map(run_one, jobs), total=len(jobs)))
print(f"{len(jobs)} OCR · {(time.time() - t0) / 60:.1f} dk")
R = pd.DataFrame(res)

# Ham görüntüye göre: korunan ve kazanılan doğru kelimeler
raw_words = R[R.yöntem == "A · ham görüntü"].set_index("belge")["_words"]
R["korunan_kelime"] = [100 * len(ws & raw_words[b]) / len(raw_words[b]) if raw_words[b] else np.nan for b, ws in zip(R.belge, R._words)]
R["kazanılan_kelime"] = [len(ws - raw_words[b]) for b, ws in zip(R.belge, R._words)]
R = R.drop(columns="_words")
R.to_csv(ROOT / "gurultu_detay.csv", index=False)

## 4. Sonuçlar

In [ ]:
summary = R.groupby("yöntem").agg(
    ocr_güveni=("güven", "mean"), kelime=("kelime", "mean"), sözlük_oranı=("sözlük_oranı", "mean"),
    korunan_kelime_yüzde=("korunan_kelime", "mean"), kazanılan_kelime=("kazanılan_kelime", "mean"),
    koyu_piksel_kaybı_yüzde=("koyu_piksel_kaybı", "mean")).round(3).reindex(list(METHODS))
summary.to_csv(ROOT / "gurultu_sonuclari.csv")
display(summary)

per_class = R.pivot_table(index="yöntem", columns="sınıf", values="sözlük_oranı", aggfunc="mean").reindex(list(METHODS))[CLASSES].round(3)
print("Sınıf bazında sözlük oranı:")
display(per_class.style.background_gradient(cmap="RdYlGn", axis=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
names = [n.split(" · ")[0] + "\n" + n.split(" · ")[1] for n in summary.index]
colors = ["#0f6e66" if n.startswith("A") else "#9fb8b4" for n in summary.index]
for ax, col, title in [(axes[0], "sözlük_oranı", "Sözlük oranı (yüksek = iyi)"),
                       (axes[1], "korunan_kelime_yüzde", "Ham görüntüdeki doğru kelimelerin korunan kısmı (%)"),
                       (axes[2], "koyu_piksel_kaybı_yüzde", "Silinen yazı pikseli (%)")]:
    bars = ax.bar(range(len(summary)), summary[col], color=colors)
    ax.bar_label(bars, fmt="%.2f" if col == "sözlük_oranı" else "%.1f", fontsize=8, padding=2)
    ax.set_xticks(range(len(summary))); ax.set_xticklabels(names, fontsize=8)
    ax.set_title(title, fontsize=11); ax.grid(axis="y", alpha=.3)
plt.suptitle("Gürültü giderme yöntemlerinin OCR'a etkisi (her sınıftan 10 validation belgesi)", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "gurultu_karsilastirma.png", dpi=120); plt.show()

In [ ]:
raw = summary.loc["A · ham görüntü"]
others = summary.drop(index="A · ham görüntü")
least_loss = others["korunan_kelime_yüzde"].fillna(0).idxmax()
best_dict = others["sözlük_oranı"].fillna(0).idxmax()
print(f"Ham görüntü: sözlük oranı {raw.sözlük_oranı:.3f}")
print(f"Metni en az yiyen yöntem: {least_loss} → ham görüntüdeki doğru kelimelerin %{others.loc[least_loss, 'korunan_kelime_yüzde']:.1f}'i korunuyor, "
      f"yazı piksellerinin %{others.loc[least_loss, 'koyu_piksel_kaybı_yüzde']:.1f}'i siliniyor")
print(f"Sözlük oranı en yüksek yöntem: {best_dict} ({others.loc[best_dict, 'sözlük_oranı']:.3f})")
better = others[others.sözlük_oranı > raw.sözlük_oranı]
if better.empty:
    print("Hiçbir yöntem ham görüntüden daha iyi okunmadı → pipeline ham görüntüyle devam eder (pilot deneyin sonucu doğrulandı).")
else:
    print("Ham görüntüden daha iyi okunan yöntemler:", ", ".join(better.index),
          "→ tüm veri setinde denenmeye aday (02'de USE_PREPROCESS yerine bu fonksiyon kullanılabilir).")

## Yorum için

- **"Korunan kelime"** danışmanın sorusunun doğrudan cevabıdır: ham görüntüde doğru okunan kelimelerin yüzde kaçı yöntem uygulandıktan sonra hâlâ okunuyor.
- **"Kazanılan kelime"** yöntemin gürültü yüzünden okunamayan kelimeleri geri kazanıp kazanmadığını gösterir. İyi bir yöntem az yiyip çok kazanmalı.
- Fourier yöntemleri görüntünün tamamına uygulanır: ince harf çizgileri de yüksek frekanslı olduğu için gürültüyle birlikte zayıflar.
- Küçük leke silme sadece harf olamayacak kadar küçük noktalara dokunduğu için yazıyı en az bozması beklenir; ama i noktası, nokta ve virgül gibi işaretleri de silebilir.

| Dosya | İçerik |
|---|---|
| `gurultu_sonuclari.csv` | Yöntem başına ortalamalar |
| `gurultu_detay.csv` | Belge × yöntem sonuçları |
| `figures/gurultu_ornek.png` | Aynı bölgenin yöntemlere göre görünümü |
| `figures/gurultu_karsilastirma.png` | Sonuç grafiği |